# 🧠🤖 Tutorial #1 — Train Your Own Model (Interactive)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/atifhalim/BrainWerks/blob/main/learning/notebooks/Tutorial1_Train_Model.ipynb)

Now you actually **train an AI model**, using the same words and ideas as the slides.

**The flow:**
1. **Step 2 — pick a Dataset** and its settings, and look inside it (a scrollable table of the raw numbers).
2. **Step 3 — pick a Model.**
3. **Step 4 — set the training settings and press 🚀 Train** — your Step 2 & 3 choices are used automatically.

### How to start
Just click **Runtime ▸ Run all** once. That's the only step — setup runs by itself, **all the code stays hidden**, and you'll only see **results**: the controls, the data table, and the training output.

*(Colab can't run a notebook automatically the moment it opens, so this one click is the single thing to do. Optional: **Runtime ▸ Change runtime type ▸ GPU** for faster training.)*

## Step 1 — Set up (run once)

In [ ]:
#@title ⚙️ Setup — installs everything (runs when you press Run all)
# Setup — installs the packages and keeps them compatible with Colab.
import subprocess, sys
print("Setting up… installing packages (~30–60s the first time).")
pkgs = ["mne", "braindecode", "ipywidgets", "pandas<3"]
res = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs],
                     capture_output=True, text=True)
IGNORE = ("dependency resolver", "moviepy", "decorator", "google-colab", "pip's ")
def _noise(l):
    return (not l.strip()) or any(k in l for k in IGNORE)
if res.returncode != 0:
    lines = [l for l in (res.stdout + res.stderr).splitlines() if not _noise(l)]
    print("\nInstall problem:\n" + "\n".join(lines[-20:]))
    raise SystemExit("Setup failed. Try Runtime ▸ Restart session, then run again.")
print("✅ Setup complete — you can ignore any Colab package notes.")

## Step 2 — Pick a **Dataset** and its settings

A **Dataset** is a stack of **Epochs (EEG)** — equal-length, labeled **clips**. Each epoch is a grid of
**Channels × Samples**, and every epoch carries one **Class**. Stacking them gives **X and y**:

* **X** = all the clips → shape **(trials, channels, time)**
* **y** = one **Class** label per clip

**The settings change with the dataset — you can only adjust what the dataset's creators didn't fix:**

* **Synthetic** — you set *everything*: **trials, channels, time (samples), classes**. *(Defaults match the slides: `(100, 3, 1024)`, 4 classes.)*
* **Real datasets** — you choose **how many Channels** to use and the **clip length (seconds)**; the number of **Classes** and the **sampling rate** are **fixed by the recording**, and the **trials** count is derived from the clip length.

*(In a full pipeline you'd also **Filter** the signal before stacking.)*

In [ ]:
#@title 📂 Dataset picker + settings (use the controls below)
import numpy as np, pandas as pd, mne, warnings
warnings.filterwarnings("ignore"); mne.set_log_level("ERROR")
from IPython.display import display, clear_output
import ipywidgets as widgets
try:
    from google.colab import output as _o; _o.enable_custom_widget_manager()
    from google.colab import data_table; data_table.enable_dataframe_formatter()
except Exception:
    pass

SELECTED = {"task": None, "model": None}
SYNTH   = {"trials": 100, "channels": 3, "times": 1024, "classes": 4}   # the slides' shape
REALCFG = {}          # per real task: {"channels": N, "clip": seconds}
_raw    = {}          # cached raw recordings (download once)

SYN_NAME = "Synthetic (random noise)"
KIND = {"Real: eyes open vs closed (alpha)": "alpha", "Real: imagine LEFT vs RIGHT hand": "motor"}
POOL = {   # preferred channels, in priority order; only those actually present are offered
    "Real: eyes open vs closed (alpha)": ["O1","Oz","O2","P3","Pz","P4","POz","PO3","PO4","P1","P2","P7","P8"],
    "Real: imagine LEFT vs RIGHT hand":  ["C3","Cz","C4","C1","C2","CP3","CP4","FC3","FC4","C5","C6","CP1","CP2"],
}
TASKS = [SYN_NAME] + list(KIND)
DESC = {
    SYN_NAME:
        "Random noise with random Class labels - there is NO pattern, so Accuracy should stay near chance "
        "(100/classes %). You set its whole shape below; the slides used 100 trials, 3 Channels, 1024 Samples, 4 classes.",
    "Real: eyes open vs closed (alpha)":
        "Real EEG, two Classes: eyes OPEN vs CLOSED. Alpha waves grow when the eyes close (occipital Channels). "
        "Choose how many Channels and the clip length; the 2 Classes and sampling rate are fixed by the recording.",
    "Real: imagine LEFT vs RIGHT hand":
        "Real EEG, two Classes: imagine the LEFT vs RIGHT hand (motor Channels). Choose how many Channels and the "
        "clip length; the 2 Classes and sampling rate are fixed by the recording. The hardest task here.",
}

def _load_raw(task):
    if task not in _raw:
        if KIND[task] == "alpha":
            d = {}
            for run, lab in [(1, 0), (2, 1)]:
                fn = mne.datasets.eegbci.load_data(1, [run], update_path=True)
                r = mne.io.read_raw_edf(fn[0], preload=True); mne.datasets.eegbci.standardize(r); d[lab] = r
            _raw[task] = (d, d[0].ch_names, float(d[0].info["sfreq"]))
        else:
            rs = []
            for run in [4, 8, 12]:
                fn = mne.datasets.eegbci.load_data(1, [run], update_path=True)
                r = mne.io.read_raw_edf(fn[0], preload=True); mne.datasets.eegbci.standardize(r); rs.append(r)
            raw = mne.concatenate_raws(rs); _raw[task] = (raw, raw.ch_names, float(raw.info["sfreq"]))
    return _raw[task]

def _pool(task):
    _, avail, _ = _load_raw(task)
    return [c for c in POOL[task] if c in avail]

def load_synthetic():
    n, c, t, k = SYNTH["trials"], SYNTH["channels"], SYNTH["times"], SYNTH["classes"]
    rng = np.random.default_rng(0)
    X = (rng.standard_normal((n, c, t)) * 8).astype("float32"); y = rng.integers(0, k, n).astype("int64")
    return dict(X=X, y=y, classes=[str(i) for i in range(k)], ch=[f"ch{i+1}" for i in range(c)], sfreq=250.0)

def load_real(task):
    cfg = REALCFG[task]; ch = _pool(task)[:cfg["channels"]]; obj, _, sf = _load_raw(task); n = int(cfg["clip"] * sf)
    if KIND[task] == "alpha":
        Xs, ys = [], []
        for lab, r in obj.items():
            d = r.copy().pick(ch).get_data() * 1e6; segs = d.shape[1] // n
            Xs += [d[:, i*n:(i+1)*n] for i in range(segs)]; ys += [lab] * segs
        return dict(X=np.stack(Xs).astype("float32"), y=np.array(ys, "int64"),
                    classes=["eyes open", "eyes closed"], ch=list(ch), sfreq=sf)
    raw = obj.copy().pick(ch); events, eid = mne.events_from_annotations(raw)
    ep = mne.Epochs(raw, events, {k: eid[k] for k in ("T1", "T2")}, tmin=0.0, tmax=cfg["clip"] - 1/sf,
                    baseline=None, preload=True)
    return dict(X=(ep.get_data() * 1e6).astype("float32"),
                y=(ep.events[:, -1] == eid["T2"]).astype("int64"),
                classes=["left hand", "right hand"], ch=list(ch), sfreq=sf)

def get_task(name):
    return load_synthetic() if name == SYN_NAME else load_real(name)

def _table(d):
    X, ch, sf = d["X"], d["ch"], d["sfreq"]; ntr, C_, T = X.shape
    flat = np.transpose(X, (0, 2, 1)).reshape(ntr*T, C_)
    trial = np.repeat(np.arange(ntr), T); samp = np.tile(np.arange(T), ntr)
    df = pd.DataFrame(flat, columns=[f"{c} (uV)" for c in ch]).round(2)
    df.insert(0, "Time (ms)", np.round(samp / sf * 1000, 1)); df.insert(0, "Sample", samp); df.insert(0, "Trial (epoch)", trial)
    return df

_out = widgets.Output()
def _render():
    with _out:
        clear_output()
        task = SELECTED["task"]
        if task != SYN_NAME:
            print("(loading real data — a moment the first time)")
        d = get_task(task); X, y, cls = d["X"], d["y"], d["classes"]
        clear_output()
        print("Selected dataset:  " + task + "\n"); print(DESC[task] + "\n")
        print("Stack every epoch -> the dataset")
        print(f"   X  (trials, channels, time) = {tuple(X.shape)}")
        print("   y  one label per epoch = [ " + ", ".join(str(int(v)) for v in y[:14]) + ", ... ]")
        print("   classes:  " + "    ".join(f"{i} = {c}" for i, c in enumerate(cls)))
        print(f"\nAll {X.shape[0]} epochs x {X.shape[2]} samples as one scrollable table"
              " (Trial = which Epoch, Sample = moment in time):")
        display(_table(d))

_W = widgets.Layout(width="540px"); _S = {"description_width": "initial"}
dd   = widgets.Dropdown(options=TASKS, description="Dataset:", layout=_W, style=_S)
s_tr = widgets.IntSlider(value=100, min=60, max=300, step=10, description="Synthetic - trials:", continuous_update=False, layout=_W, style=_S)
s_ch = widgets.IntSlider(value=3, min=2, max=8, step=1, description="Synthetic - channels:", continuous_update=False, layout=_W, style=_S)
s_ti = widgets.IntSlider(value=1024, min=512, max=2048, step=64, description="Synthetic - time (samples):", continuous_update=False, layout=_W, style=_S)
s_cl = widgets.IntSlider(value=4, min=2, max=4, step=1, description="Synthetic - classes:", continuous_update=False, layout=_W, style=_S)
r_ch = widgets.IntSlider(value=3, min=2, max=3, step=1, description="Channels to use:", continuous_update=False, layout=_W, style=_S)
r_cl = widgets.FloatSlider(value=2.0, min=2.0, max=4.0, step=0.5, description="Clip length (sec):", continuous_update=False, layout=_W, style=_S)
info = widgets.HTML()
box  = widgets.VBox([]); _guard = {"on": False}

def _on_synth(_):
    if _guard["on"]: return
    SYNTH.update(trials=s_tr.value, channels=s_ch.value, times=s_ti.value, classes=s_cl.value); _render()

def _on_real(_):
    if _guard["on"]: return
    REALCFG[dd.value] = {"channels": r_ch.value, "clip": r_cl.value}; _render()

def _on_dataset(_=None):
    task = dd.value; SELECTED["task"] = task; _guard["on"] = True
    if task == SYN_NAME:
        box.children = [s_tr, s_ch, s_ti, s_cl]
    else:
        p = _pool(task); r_ch.max = max(2, len(p))
        default = min(6 if KIND[task] == "alpha" else 3, r_ch.max)
        cfg = REALCFG.setdefault(task, {"channels": default, "clip": 2.0})
        r_ch.value = min(cfg["channels"], r_ch.max); r_cl.value = cfg["clip"]
        REALCFG[task] = {"channels": r_ch.value, "clip": r_cl.value}
        _, _, sf = _load_raw(task)
        info.value = ("<i>Fixed by the dataset creators: 2 Classes, sampling rate " + str(int(sf))
                      + " Hz. Trials are derived from the clip length.</i>")
        box.children = [r_ch, r_cl, info]
    _guard["on"] = False
    _render()

for w in (s_tr, s_ch, s_ti, s_cl): w.observe(_on_synth, "value")
for w in (r_ch, r_cl): w.observe(_on_real, "value")
dd.observe(_on_dataset, "value")
display(widgets.VBox([dd, box, _out])); _on_dataset()

## Step 3 — Pick a **Model**

The **Model** is the AI that learns the mapping from **X** (brain clips) to **y** (the **Class**).
braindecode ships 65+ of them; here are three good ones. Pick one — it will be used in Step 4.

In [ ]:
#@title 🧠 Model picker (use the dropdown below)
import torch, ipywidgets as widgets
from braindecode.models import ShallowFBCSPNet, Deep4Net, EEGNet
from braindecode import EEGClassifier
from braindecode.util import set_random_seeds
from skorch.dataset import ValidSplit
from sklearn.model_selection import train_test_split

MODELS = {
    "ShallowFBCSPNet (recommended)": "Shallow",
    "Deep4Net (bigger)":             "Deep",
    "EEGNet (compact)":              "EEGNet",
}
MDESC = {
    "ShallowFBCSPNet (recommended)":
        "A small, fast network - braindecode's beginner default. It learns X -> y well even with little data, so start here.",
    "Deep4Net (bigger)":
        "A deeper, bigger network. It can capture more, but usually needs more data and more Training Passes before it beats Shallow.",
    "EEGNet (compact)":
        "A compact, efficient network with very few parts - quick to Train (.fit) and a nice one to compare against Shallow.",
}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def build_and_train(task_name, model_label, lr, epochs, batch_size, log=print):
    set_random_seeds(20240205, cuda=(DEVICE == "cuda"))
    d = get_task(task_name); X, y, classes = d["X"], d["y"], d["classes"]
    n_cls, ch, T = len(classes), X.shape[1], X.shape[2]
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=1, stratify=y)
    key = MODELS[model_label]
    Model = {"Shallow": ShallowFBCSPNet, "Deep": Deep4Net, "EEGNet": EEGNet}[key]
    mod = (Model(n_chans=ch, n_outputs=n_cls, n_times=T, final_conv_length="auto")
           if key in ("Shallow", "Deep") else Model(n_chans=ch, n_outputs=n_cls, n_times=T))
    clf = EEGClassifier(mod, criterion=torch.nn.CrossEntropyLoss,
                        optimizer=torch.optim.Adam, optimizer__lr=lr,
                        batch_size=batch_size, max_epochs=epochs,
                        train_split=ValidSplit(0.2), device=DEVICE, verbose=1)
    clf.fit(Xtr, ytr)
    return clf, clf.score(Xte, yte), n_cls, classes, (len(Xtr), len(Xte))

def pick_model(model):
    SELECTED["model"] = model
    print("Selected model:  " + model + "\n"); print(MDESC[model])

print("Training will run on:", DEVICE.upper(), "(tip: Runtime > Change runtime type > GPU for speed)\n")
widgets.interact(pick_model,
    model=widgets.Dropdown(options=list(MODELS), description="Model:",
        layout=widgets.Layout(width="540px"), style={"description_width": "initial"}));

## Step 4 — Set the settings and **Train**

Your **Dataset** (Step 2) and **Model** (Step 3) are already chosen — just set three knobs and press **🚀 Train**.

* **Training passes (epochs)** — one **Epoch (training)** = one full pass through all the training clips. *(The slides used 10.)* ⚠️ This is a **different** meaning of "epoch" than an **Epoch (EEG)** clip!
* **Batch size** — how many clips the model looks at before each little adjustment. *(Not covered in the slides; smaller = more frequent updates.)*
* **Learning rate** — *(also not in the slides):* how **big a step** the model takes each time it adjusts during **Train (.fit)**. Too **high** → it overshoots and never settles; too **low** → it learns painfully slowly. The default `0.000625` is a solid start.

Pressing **Train** splits the data into **Training** clips (to learn from) and held-out **Validation / Test** clips (to score honestly). You'll see the per-pass table, then the **Accuracy** on clips the model never saw.

In [ ]:
#@title 🚀 Training controls (set the sliders, then press Train)
import ipywidgets as widgets, matplotlib.pyplot as plt
from IPython.display import display, clear_output

w_lr = widgets.FloatLogSlider(value=6.25e-4, base=10, min=-4, max=-2, step=0.1,
        description="Learning rate:", readout_format=".4f",
        style={"description_width": "initial"}, layout=widgets.Layout(width="540px"))
w_ep = widgets.IntSlider(value=10, min=1, max=100, step=1, description="Training passes (epochs):",
        style={"description_width": "initial"}, layout=widgets.Layout(width="540px"))
w_bs = widgets.Dropdown(options=[8, 16, 32, 64], value=32, description="Batch size:",
        style={"description_width": "initial"})
btn = widgets.Button(description="🚀 Train", button_style="success")
out = widgets.Output()

def _run(_):
    with out:
        clear_output()
        task = SELECTED.get("task") or TASKS[0]
        model = SELECTED.get("model") or list(MODELS)[0]
        print(f"Dataset (from Step 2): {task}")
        print(f"Model   (from Step 3): {model}\n")
        print("Training... (a real dataset downloads a small sample the first time)\n")
        try:
            clf, acc, n_cls, classes, (ntr, nte) = build_and_train(
                task, model, w_lr.value, w_ep.value, w_bs.value)
            chance = 1.0 / n_cls
            print("\n-------- RESULT --------")
            print(f"trained on {ntr} clips, tested on {nte} unseen (Validation/Test) clips")
            print(f"TEST ACCURACY = {acc*100:.1f}%   (chance = {chance*100:.0f}%)")
            if acc >= chance + 0.15:   print("Well above chance - it really learned the pattern!")
            elif acc >= chance + 0.05: print("A bit above chance - try more passes or another Model.")
            else:                      print("Around chance - like the random-noise task, no real pattern was found.")
            fig, ax = plt.subplots(figsize=(5, 2.6))
            ax.bar(["chance", "this model"], [chance*100, acc*100], color=["#9aa0b5", "#22a06b"])
            ax.set_ylabel("Accuracy (%)"); ax.set_ylim(0, 100)
            for i, v in enumerate([chance*100, acc*100]): ax.text(i, v+2, f"{v:.0f}%", ha="center")
            plt.tight_layout(); plt.show()
        except Exception as e:
            print("Something went wrong:", type(e).__name__, str(e)[:300])

btn.on_click(_run)
display(widgets.VBox([w_lr, w_ep, w_bs, btn, out]))

## Try these experiments 🔬

* **Eyes open vs closed + ShallowFBCSPNet, ~10–25 passes** → Accuracy should jump well above 50%.
* **Synthetic (random) task** → stays near chance no matter what: no pattern, no learning.
* **Imagine LEFT vs RIGHT hand** → the hard one; add Channels, try more passes, or a different Model.
* **On a real dataset, add more Channels or a longer clip** → does the Accuracy change?
* **Nudge the Learning rate** up and down to feel the sweet spot.

Every run splits, trains, and scores on unseen clips — the honest measure of learning.